# 🧠 Multimodal Sign Language Recognition Pipeline
### Pose (MediaPipe) + Visual (MobileNetV2) + Temporal (BiLSTM)

This notebook implements the training pipeline for a sign language recognition model inspired by the INCLUDE dataset methodology. It combines skeletal keypoints and visual features to capture both motion and context.

**Pipeline Overview:**
1. **Extraction**: Unzip INCLUDE dataset files.
2. **Feature Engineering**: Extract frame-wise pose keypoints and MobileNetV2 features.
3. **Fusion**: Combine structured pose data with visual context.
4. **Modeling**: Use a Bidirectional LSTM to capture temporal dynamics.
5. **Training**: Phase-wise optimization with learning rate scheduling.

In [ ]:
# 1. Setup & Dependencies
!pip install mediapipe opencv-python torch torchvision tqdm scikit-learn matplotlib -q

import os
import cv2
import zipfile
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import mediapipe as mp
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset Preparation
Assuming you have your INCLUDE dataset zip files ready.

In [ ]:
# --- Configuration ---
ZIP_FILES = ["include_train.zip", "include_val.zip"] # Update with your filenames
EXTRACT_DIR = "include_videos"
FEATURE_DIR = "extracted_features"
MAX_FRAMES = 30 # Fixed sequence length
IMAGE_SIZE = 224

# Extract zips
os.makedirs(EXTRACT_DIR, exist_ok=True)
for zip_path in ZIP_FILES:
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(EXTRACT_DIR)
    else:
        print(f"Warning: {zip_path} not found.")

## 3. Feature Extraction Pipeline
We extract:
1. **Pose Features**: (x, y, z, visibility) for 33 pose landmarks + 21 hand landmarks.
2. **Visual Features**: 1280-dim vector from MobileNetV2 (last pooling layer).

In [ ]:
# Initialize MediaPipe Holistic
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, min_detection_confidence=0.5)

# Initialize MobileNetV2
mobilenet = models.mobilenet_v2(weights="IMAGENET1K_V1").to(device)
mobilenet.classifier = nn.Identity() # Remove classification head
mobilenet.eval()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_features_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    pose_features = []
    visual_features = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # 1. Pose Extraction
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image_rgb)
        
        # Construct pose vector
        kp = []
        if results.pose_landmarks:
            for res in results.pose_landmarks.landmark:
                kp.extend([res.x, res.y, res.z, res.visibility])
        else: kp.extend([0] * (33 * 4))
            
        if results.left_hand_landmarks:
            for res in results.left_hand_landmarks.landmark:
                kp.extend([res.x, res.y, res.z])
        else: kp.extend([0] * (21 * 3))
            
        if results.right_hand_landmarks:
            for res in results.right_hand_landmarks.landmark:
                kp.extend([res.x, res.y, res.z])
        else: kp.extend([0] * (21 * 3))
        
        pose_features.append(kp)
        
        # 2. Visual Extraction
        with torch.no_grad():
            input_tensor = preprocess(image_rgb).unsqueeze(0).to(device)
            vis_feat = mobilenet(input_tensor).squeeze().cpu().numpy()
            visual_features.append(vis_feat)
            
    cap.release()
    return np.array(pose_features), np.array(visual_features)

def process_dataset(video_root, output_root):
    os.makedirs(output_root, exist_ok=True)
    video_paths = glob.glob(os.path.join(video_root, "**/*.mp4"), recursive=True)
    
    for v_path in tqdm(video_paths, desc="Extracting features"):
        save_name = os.path.basename(v_path).replace(".mp4", ".npz")
        label = os.path.basename(os.path.dirname(v_path))
        label_dir = os.path.join(output_root, label)
        os.makedirs(label_dir, exist_ok=True)
        
        save_path = os.path.join(label_dir, save_name)
        if os.path.exists(save_path): continue
        
        pose, visual = extract_features_from_video(v_path)
        np.savez_compressed(save_path, pose=pose, visual=visual)

# Run extraction (this might take a while)
# process_dataset(EXTRACT_DIR, FEATURE_DIR)

## 4. Dataset & DataLoader
We use uniform sampling to normalize sequence lengths.

In [ ]:
class SLRDataset(Dataset):
    def __init__(self, feature_dir, max_frames=30):
        self.max_frames = max_frames
        self.files = glob.glob(os.path.join(feature_dir, "**/*.npz"), recursive=True)
        self.labels = sorted(list(set([os.path.basename(os.path.dirname(f)) for f in self.files])))
        self.label_to_idx = {l: i for i, l in enumerate(self.labels)}
        
    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        f_path = self.files[idx]
        data = np.load(f_path)
        pose, visual = data['pose'], data['visual']
        label = self.label_to_idx[os.path.basename(os.path.dirname(f_path))]
        
        # Uniform Sampling / Padding
        n_frames = pose.shape[0]
        if n_frames > self.max_frames:
            indices = np.linspace(0, n_frames - 1, self.max_frames, dtype=int)
            pose = pose[indices]
            visual = visual[indices]
        elif n_frames < self.max_frames:
            pad_size = self.max_frames - n_frames
            pose = np.pad(pose, ((0, pad_size), (0, 0)), mode='constant')
            visual = np.pad(visual, ((0, pad_size), (0, 0)), mode='constant')
            
        return torch.FloatTensor(pose), torch.FloatTensor(visual), torch.tensor(label)

## 5. Model Architecture
Combining Pose and CNN features followed by a BiLSTM.

In [ ]:
class MultimodalBiLSTM(nn.Module):
    def __init__(self, pose_dim, cnn_dim, hidden_dim, n_classes, n_layers=2):
        super().__init__()
        # Projections to align dimensions
        self.pose_proj = nn.Sequential(nn.Linear(pose_dim, 256), nn.ReLU(), nn.Dropout(0.3))
        self.cnn_proj = nn.Sequential(nn.Linear(cnn_dim, 256), nn.ReLU(), nn.Dropout(0.3))
        
        # Temporal Model
        self.lstm = nn.LSTM(input_size=512, # 256 + 256
                            hidden_size=hidden_dim, 
                            num_layers=n_layers, 
                            batch_first=True, 
                            bidirectional=True, 
                            dropout=0.5)
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256), # * 2 for Bidirectional
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes)
        )
        
    def forward(self, pose_seq, cnn_seq):
        # pose_seq: (B, T, D_pose), cnn_seq: (B, T, D_cnn)
        p_feat = self.pose_proj(pose_seq)
        c_feat = self.cnn_proj(cnn_seq)
        
        # Fusion
        fused = torch.cat([p_feat, c_feat], dim=-1)
        
        # LSTM
        lstm_out, _ = self.lstm(fused)
        
        # Global Average Pooling over time (or use last hidden state)
        out = torch.mean(lstm_out, dim=1)
        
        # Classify
        logits = self.classifier(out)
        return logits

# Dimensions based on MediaPipe Holistic output
POSE_DIM = (33 * 4) + (21 * 3) + (21 * 3) # Pose + Left Hand + Right Hand = 258
CNN_DIM = 1280 # MobileNetV2 output
HIDDEN_DIM = 512
N_CLASSES = 263 # Full INCLUDE classes

## 6. Training Script

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_acc = 0
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for p_seq, c_seq, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            p_seq, c_seq, labels = p_seq.to(device), c_seq.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(p_seq, c_seq)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Validation
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for p_seq, c_seq, labels in val_loader:
                p_seq, c_seq, labels = p_seq.to(device), c_seq.to(device), labels.to(device)
                outputs = model(p_seq, c_seq)
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        avg_loss = total_loss / len(train_loader)
        scheduler.step(avg_loss)
        
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(acc)
        
        print(f"Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")
        
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "best_multimodal_model.pth")
            
    return history

# To run training:
# dataset = SLRDataset(FEATURE_DIR)
# train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
# model = MultimodalBiLSTM(POSE_DIM, CNN_DIM, HIDDEN_DIM, N_CLASSES).to(device)
# history = train_model(model, train_loader, val_loader)